# سبق 11 - ماڈل کانٹیکسٹ پروٹوکول (MCP)

The **ماڈل کانٹیکسٹ پروٹوکول (MCP)** ایک کھلا معیار ہے جو ایجنٹس کو رن ٹائم پر ٹولز، وسائل، اور ڈیٹا ذرائع کو متحرک طور پر دریافت کرنے اور استعمال کرنے کے قابل بناتا ہے۔ کسی ایجنٹ میں ٹولز کو ہارڈ کوڈ کرنے کے بجائے، MCP ایجنٹس کو ایسے بیرونی سرورز سے جڑنے دیتا ہے جو مانگ پر صلاحیتیں ظاہر کرتے ہیں۔

In this lesson, you'll learn:
- MCP کیا ہے اور ایجنٹ سسٹمز کے لیے یہ کیوں اہم ہے
- MCP کلائنٹ-سرور آرکیٹیکچر کیسے کام کرتا ہے
- ایسے ایجنٹس بنانے کا طریقہ جو MCP-طرز کی ٹول ڈسکوری استعمال کریں


## سیٹ اپ

**ضروریات:**
- Azure AI Foundry پروجیکٹ جس میں ایک تعینات شدہ ماڈل ہو
- تصدیق کے لیے `az login` چلائیں


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

\u200F## ماڈل کانٹیکسٹ پروٹوکول (MCP) کیا ہے؟

MCP بیرونی ٹولز اور ڈیٹا ذرائع کو دریافت کرنے اور ان کے ساتھ بات چیت کرنے کے لئے ایک معیاری طریقہ متعین کرتا ہے:

- **MCP Server**: معیاری پروٹوکول کے ذریعے ٹولز، وسائل، اور پرامپٹس کو ظاہر کرتا ہے
- **MCP Client**: ایجنٹ رن ٹائم جو سرورز سے منسلک ہوتا ہے اور دستیاب صلاحیتوں کو دریافت کرتا ہے
- **Dynamic Discovery**: ایجنٹس کو ہارڈ کوڈڈ ٹولز کی ضرورت نہیں — وہ رن ٹائم پر دستیاب چیزوں کا پتہ لگاتے ہیں

یہ وسیع پذیر ایجنٹ سسٹمز بنانے کے لئے طاقتور ہے جہاں نئی صلاحیتیں ایجنٹ کے کوڈ میں تبدیلی کئے بغیر شامل کی جا سکتی ہیں۔


## MCP کیسے کام کرتا ہے

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ایجنٹ (MCP client) ایک MCP سرور سے جڑتا ہے
2. سرور دستیاب ٹولز اور ان کے اسکیموں کی فہرست کے ساتھ جواب دیتا ہے
3. ایجنٹ پھر اپنی استدلال کے دوران کسی بھی دریافت شدہ ٹول کو کال کر سکتا ہے
4. نتائج اسی پروٹوکول کے ذریعے واپس آتے ہیں


## MCP ٹول کی دریافت کی نقالی

چونکہ ایک حقیقی MCP سرور کے لیے ایک چلتا ہوا سرور پروسس ضروری ہوتا ہے، ہم اس پیٹرن کو ایسے `@tool` فنکشنز کی مدد سے پیش کریں گے جو MCP سے منسلک رہائش سروس کی فراہم کردہ چیزوں کی نقالی کرتے ہیں۔

پروڈکشن میں، یہ ٹولز مقامی طور پر تعریف کیے جانے کے بجائے ایک MCP سرور سے متحرک طور پر دریافت کیے جائیں گے۔


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP طرز کے اوزاروں کے ساتھ ایک ایجنٹ بنانا


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP پروڈکشن میں

ایک پروڈکشن ماحول میں، MCP طاقتور پیٹرنز کو ممکن بناتا ہے:

- **متحرک ٹول دریافت**: ایجنٹس MCP سرورز سے کنیکٹ ہوتے ہیں اور رن ٹائم پر ٹولز دریافت کرتے ہیں
- **جُدا شدہ فنِ تعمیر**: ٹول فراہم کرنے والے ایجنٹ سے آزادانہ طور پر اپڈیٹ کر سکتے ہیں
- **تنظیموں کے مابین اشتراک**: ٹیمیں MCP سرورز کے ذریعے ایسی صلاحیتیں ظاہر کر سکتی ہیں جنہیں کوئی بھی ایجنٹ استعمال کر سکتا ہے
- **Microsoft Agent Framework کی حمایت**: MAF میں `mcp` انٹیگریشن کے ذریعے بلٹ ان MCP کلائنٹ کی حمایت موجود ہے

حقیقی MCP سرور کو MAF کے ساتھ استعمال کرنے کے لیے، آپ `hosted_mcp_tool()` یا MCP کلائنٹ انٹیگریشن کے ذریعے کنیکٹ کریں گے۔

**مزید جانیں:**
- [MCP کی وضاحت](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework میں MCP کی حمایت](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## خلاصہ

اس سبق میں، آپ نے سیکھا:
- **MCP** ایجنٹس اور ٹول فراہم کنندگان کے درمیان متحرک ٹول دریافت کے لیے ایک کھلا معیار ہے
- یہ **کلائنٹ-سرور فن تعمیر** ایجنٹس کو رن ٹائم میں صلاحیتیں دریافت کرنے کی اجازت دیتی ہے
- MCP **قابلِ توسیع، علیحدہ ایجنٹ سسٹمز** کو فعال بناتا ہے جہاں اوزار کو کوڈ تبدیلیوں کے بغیر شامل کیا جا سکتا ہے
- Microsoft Agent Framework پیداواری استعمال کے لیے **بلٹ-ان MCP سپورٹ** فراہم کرتا ہے


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
دستبرداری:
یہ دستاویز AI ترجمہ سروس Co‑op Translator (https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ مہربانی نوٹ کریں کہ خودکار تراجم میں غلطیاں یا عدمِ درستگی شامل ہو سکتی ہیں۔ اصل دستاویز اس کی مادری زبان میں ماخذِ حقّیق سمجھا جائے۔ اہم معلومات کے لیے پیشہ ور، انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
